In [ ]:
# ============================================================
# 03_model_training_evaluation.ipynb
# Sex-stratified risk-classifier training, selection, and
# validation for AI underwriting (model-risk controls).
#
# For each sex: 7-algorithm comparison (XGBoost, CatBoost, LightGBM,
# RandomForest, ExtraTrees, SVM, MLP) tuned by Optuna (TPE),
# stratified 70/30 split, 5-fold CV; accuracy-first threshold
# (F1 >= 0.48); DeLong AUC tests vs XGBoost; bootstrap 95% AUC CI;
# Platt (sigmoid) calibration with Brier/ECE before-after.
#
# Input : data/processed/df_male.csv, df_female.csv, feature_list.csv
# Output: data/processed/final_pipeline_{sex}.pkl,
#         data/processed/scored_{sex}.csv (calibrated risk_prob),
#         data/processed/models_{sex}/{xgb,lgb,cat}_pipe.pkl,
#         results/tables/performance_{sex}.csv, delong_{sex}.csv,
#         results/tables/calibration.csv, final_model_summary.csv,
#         results/figures/roc_{sex}.(png|pdf),
#         results/figures/calibration_{sex}.(png|pdf),
#         results/figures/decision_curve_male_female.(png|pdf)
#
# Reproducibility: SEED=42. Set env N_TRIALS to control Optuna trials
# (paper used 100; default here 100). Figures: grayscale, dpi 600,
# PNG + PDF, no titles/captions.
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, average_precision_score,
                             brier_score_loss)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler
from scipy import stats
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_TRIALS = int(os.environ.get("N_TRIALS", "100"))  # paper used 100
F1_MIN = 0.48
N_BOOT = 1000
np.random.seed(SEED)

# ---- Paths ----
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

# ---- Grayscale style ----
GRAY = ["#000000", "#444444", "#777777", "#A0A0A0", "#C4C4C4", "#8C8C8C", "#D9D9D9"]
LS = ["-", "--", "-.", ":", "-", "--", "-."]
sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 11, "axes.edgecolor": "black",
    "axes.linewidth": 0.8, "axes.grid": True, "grid.color": "0.88",
    "grid.linewidth": 0.6, "image.cmap": "Greys",
})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

cat_col = ["HE_obe","BO1_1","BO1_2","BO1_3","BD1_11","BD2_1","BS3_1","BE3_71",
           "BE3_75","BE3_81","BE3_91","pa_aerobic","L_BR_FQ","BP1","mh_stress",
           "incm","ho_incm","edu","BH1"]
num_col = ["HE_BMI","HE_wc","HE_wt","N_SUGAR","N_CHO","N_EN","age"]
feature_col = cat_col + num_col
TARGET = "HE_DM_HbA1c"

def make_prep():
    return ColumnTransformer([
        ("num", StandardScaler(), num_col),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_col),
    ])

def optimize_threshold(y_true, y_prob, f1_min=F1_MIN):
    best_t, best_acc = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.01):
        yp = (y_prob >= t).astype(int)
        if f1_score(y_true, yp, zero_division=0) >= f1_min:
            acc = accuracy_score(y_true, yp)
            if acc > best_acc:
                best_acc, best_t = acc, t
    if best_acc == 0.0:
        fpr, tpr, thr = roc_curve(y_true, y_prob)
        best_t = thr[np.argmax(tpr - fpr)]
    return round(float(best_t), 2)

def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1
    e, n = 0.0, len(y_true)
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        e += (m.sum() / n) * abs(y_true[m].mean() - y_prob[m].mean())
    return e

def delong_var(y, p):
    y = np.asarray(y); p = np.asarray(p)
    pos, neg = p[y == 1], p[y == 0]
    n1, n0 = len(pos), len(neg)
    V10 = np.array([(pos[i] > neg).mean() for i in range(n1)])
    V01 = np.array([(neg[j] < pos).mean() for j in range(n0)])
    var = V10.var(ddof=1) / n1 + V01.var(ddof=1) / n0
    return roc_auc_score(y, p), var

def bootstrap_auc_ci(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y); p = np.asarray(p); n = len(y); aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

# ---- Optuna objective builders (return CV mean-std of F1) ----
def cv_f1(model, X, y):
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    s = cross_val_score(model, X, y, cv=cv, scoring="f1")
    return s.mean() - s.std()

def tune(name, Xtr, ytr):
    pos_ratio = (ytr == 0).sum() / (ytr == 1).sum()
    def obj(trial):
        if name == "XGBoost":
            m = xgb.XGBClassifier(
                n_estimators=trial.suggest_int("n_estimators", 100, 800),
                max_depth=trial.suggest_int("max_depth", 3, 10),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                subsample=trial.suggest_float("subsample", 0.6, 1.0),
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
                gamma=trial.suggest_float("gamma", 0.0, 1.0),
                reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
                reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
                scale_pos_weight=pos_ratio, eval_metric="logloss",
                random_state=SEED, n_jobs=-1)
        elif name == "CatBoost":
            m = CatBoostClassifier(
                iterations=trial.suggest_int("iterations", 100, 800),
                depth=trial.suggest_int("depth", 3, 10),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                scale_pos_weight=pos_ratio, verbose=0, random_seed=SEED)
        elif name == "LightGBM":
            m = lgb.LGBMClassifier(
                n_estimators=trial.suggest_int("n_estimators", 100, 800),
                max_depth=trial.suggest_int("max_depth", 3, 10),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                num_leaves=trial.suggest_int("num_leaves", 20, 150),
                subsample=trial.suggest_float("subsample", 0.6, 1.0),
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                min_child_samples=trial.suggest_int("min_child_samples", 5, 50),
                class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)
        elif name == "RandomForest":
            m = RandomForestClassifier(
                n_estimators=trial.suggest_int("n_estimators", 100, 600),
                max_depth=trial.suggest_int("max_depth", 3, 20),
                min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
                max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
                class_weight="balanced", random_state=SEED, n_jobs=-1)
        elif name == "ExtraTrees":
            m = ExtraTreesClassifier(
                n_estimators=trial.suggest_int("n_estimators", 100, 600),
                max_depth=trial.suggest_int("max_depth", 3, 20),
                min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
                max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
                class_weight="balanced", random_state=SEED, n_jobs=-1)
        elif name == "SVM":
            m = SVC(C=trial.suggest_float("C", 1e-2, 100.0, log=True),
                    gamma=trial.suggest_categorical("gamma", ["scale", "auto"]),
                    kernel=trial.suggest_categorical("kernel", ["rbf", "poly"]),
                    class_weight="balanced", probability=True, random_state=SEED)
        else:  # MLP
            nl = trial.suggest_int("n_layers", 1, 3)
            layers = tuple(trial.suggest_int(f"u{i}", 32, 256) for i in range(nl))
            m = MLPClassifier(hidden_layer_sizes=layers,
                              activation=trial.suggest_categorical("activation", ["relu", "tanh"]),
                              alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
                              learning_rate_init=trial.suggest_float("lr", 1e-4, 1e-1, log=True),
                              max_iter=300, random_state=SEED)
        return cv_f1(m, Xtr, ytr)
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
    study.optimize(obj, n_trials=N_TRIALS)
    bp = study.best_params
    # rebuild best estimator
    if name == "XGBoost":
        est = xgb.XGBClassifier(**bp, scale_pos_weight=pos_ratio, eval_metric="logloss",
                                random_state=SEED, n_jobs=-1)
    elif name == "CatBoost":
        est = CatBoostClassifier(**bp, scale_pos_weight=pos_ratio, verbose=0, random_seed=SEED)
    elif name == "LightGBM":
        est = lgb.LGBMClassifier(**bp, class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)
    elif name == "RandomForest":
        est = RandomForestClassifier(**bp, class_weight="balanced", random_state=SEED, n_jobs=-1)
    elif name == "ExtraTrees":
        est = ExtraTreesClassifier(**bp, class_weight="balanced", random_state=SEED, n_jobs=-1)
    elif name == "SVM":
        est = SVC(**bp, class_weight="balanced", probability=True, random_state=SEED)
    else:
        nl = bp["n_layers"]; layers = tuple(bp[f"u{i}"] for i in range(nl))
        est = MLPClassifier(hidden_layer_sizes=layers, activation=bp["activation"],
                            alpha=bp["alpha"], learning_rate_init=bp["lr"],
                            max_iter=300, random_state=SEED)
    return est

ALGOS = ["XGBoost", "CatBoost", "LightGBM", "RandomForest", "ExtraTrees", "SVM", "MLP"]
# Optional subset override for quick runs (e.g. ALGOS="XGBoost,LightGBM,CatBoost");
# the full 7-algorithm comparison is the default used for the paper.
if os.environ.get("ALGOS"):
    ALGOS = [a.strip() for a in os.environ["ALGOS"].split(",") if a.strip()]
summary_rows, calib_rows, dca_store = [], [], {}

for sex_name, fname in [("male", "df_male.csv"), ("female", "df_female.csv")]:
    print(f"\n==================== {sex_name.upper()} ====================")
    d = pd.read_csv(PROC / fname)
    X, y = d[feature_col], d[TARGET]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
    prep = make_prep()
    Xtr_p = prep.fit_transform(Xtr); Xte_p = prep.transform(Xte)
    yte_arr = yte.values

    results = {}
    for name in ALGOS:
        est = tune(name, Xtr_p, ytr)
        est.fit(Xtr_p, ytr)
        prob = est.predict_proba(Xte_p)[:, 1]
        thr = optimize_threshold(yte_arr, prob)
        yp = (prob >= thr).astype(int)
        cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
        cvf1 = cross_val_score(est, Xtr_p, ytr, cv=cv, scoring="f1")
        results[name] = dict(
            Model=name, Accuracy=round(accuracy_score(yte, yp), 4),
            Precision=round(precision_score(yte, yp, zero_division=0), 4),
            Recall=round(recall_score(yte, yp), 4),
            F1=round(f1_score(yte, yp, zero_division=0), 4),
            AUC=round(roc_auc_score(yte, prob), 4),
            PR_AUC=round(average_precision_score(yte, prob), 4),
            Brier=round(brier_score_loss(yte, prob), 4),
            Threshold=thr, CV_F1=f"{cvf1.mean():.4f} ± {cvf1.std():.4f}",
            _prob=prob, _est=est)
        print(f"  {name:13s} AUC={results[name]['AUC']:.4f}  F1={results[name]['F1']:.4f}")

    # performance table
    perf = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")}
                         for r in results.values()]).sort_values("Accuracy", ascending=False)
    perf.to_csv(TAB / f"performance_{sex_name}.csv", index=False)

    # DeLong XGBoost vs others
    dl_rows = []
    axgb, vxgb = delong_var(yte_arr, results["XGBoost"]["_prob"])
    for name in ALGOS:
        if name == "XGBoost":
            continue
        ao, vo = delong_var(yte_arr, results[name]["_prob"])
        z = (axgb - ao) / np.sqrt(vxgb + vo + 1e-12)
        p = 2 * (1 - stats.norm.cdf(abs(z)))
        dl_rows.append(dict(Comparison=f"XGBoost vs {name}", AUC_XGB=round(axgb, 4),
                            AUC_other=round(ao, 4), z=round(z, 3), p_value=round(p, 4),
                            sig="***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"))
    pd.DataFrame(dl_rows).to_csv(TAB / f"delong_{sex_name}.csv", index=False)

    # ---- Final model = XGBoost; build raw-feature pipelines ----
    best_params = {k: v for k, v in results["XGBoost"]["_est"].get_params().items()}
    pipe_xgb = Pipeline([("prep", make_prep()), ("clf", xgb.XGBClassifier(**best_params))]).fit(Xtr, ytr)
    # calibrated model for scoring (Platt / sigmoid)
    cal = CalibratedClassifierCV(pipe_xgb, method="sigmoid", cv=5).fit(Xtr, ytr)
    thr_xgb = results["XGBoost"]["Threshold"]

    # calibration before/after on test
    prob_unc = pipe_xgb.predict_proba(Xte)[:, 1]
    prob_cal = cal.predict_proba(Xte)[:, 1]
    calib_rows += [
        dict(Sex=sex_name, Stage="Pre-calibration",
             Brier=round(brier_score_loss(yte, prob_unc), 4), ECE=round(ece_score(yte_arr, prob_unc), 4)),
        dict(Sex=sex_name, Stage="Post-calibration",
             Brier=round(brier_score_loss(yte, prob_cal), 4), ECE=round(ece_score(yte_arr, prob_cal), 4)),
    ]

    # bootstrap AUC CI (XGBoost, test)
    lo, hi = bootstrap_auc_ci(yte_arr, prob_unc)
    summary_rows.append(dict(
        Sex=sex_name, Model="XGBoost", AUC=results["XGBoost"]["AUC"],
        AUC_95CI=f"[{lo:.3f}, {hi:.3f}]", PR_AUC=results["XGBoost"]["PR_AUC"],
        Brier=results["XGBoost"]["Brier"], Threshold=thr_xgb, CV_F1=results["XGBoost"]["CV_F1"]))

    # ---- Score FULL sex dataset with calibrated model (for lift/DiCE) ----
    d_scored = d.copy()
    d_scored["risk_prob"] = cal.predict_proba(X)[:, 1]
    d_scored.to_csv(PROC / f"scored_{sex_name}.csv", index=False)

    # ---- Save pipelines/artifacts ----
    joblib.dump(dict(pipe=pipe_xgb, calibrated=cal, threshold=thr_xgb,
                     feature_col=feature_col, cat_col=cat_col, num_col=num_col),
                PROC / f"final_pipeline_{sex_name}.pkl")
    mdir = PROC / f"models_{sex_name}"; mdir.mkdir(exist_ok=True)
    joblib.dump(pipe_xgb, mdir / "xgb_pipe.pkl")
    for nm, key in [("lgb", "LightGBM"), ("cat", "CatBoost")]:
        if key not in results:
            continue
        bp = {k: v for k, v in results[key]["_est"].get_params().items()}
        if key == "LightGBM":
            clf = lgb.LGBMClassifier(**bp)
        else:
            clf = CatBoostClassifier(**bp)
        pj = Pipeline([("prep", make_prep()), ("clf", clf)]).fit(Xtr, ytr)
        joblib.dump(pj, mdir / f"{nm}_pipe.pkl")

    # ---- Figure: ROC (7 algorithms, grayscale + linestyles) ----
    fig, ax = plt.subplots(figsize=(5.2, 5.0))
    for i, name in enumerate(ALGOS):
        fpr, tpr, _ = roc_curve(yte, results[name]["_prob"])
        ax.plot(fpr, tpr, color=GRAY[i % len(GRAY)], linestyle=LS[i % len(LS)],
                lw=1.6, label=f"{name} ({results[name]['AUC']:.3f})")
    ax.plot([0, 1], [0, 1], color="0.6", lw=0.8, linestyle=(0, (1, 1)))
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.legend(loc="lower right", fontsize=8, frameon=False)
    save_fig(fig, f"roc_{sex_name}")

    # ---- Figure: calibration before/after Platt (XGBoost) ----
    fig, ax = plt.subplots(figsize=(4.6, 4.4))
    ax.plot([0, 1], [0, 1], color="0.6", lw=0.8, linestyle=(0, (1, 1)), label="Perfect")
    for prob, lab, mk, gc in [(prob_unc, "Uncalibrated", "o", GRAY[2]),
                              (prob_cal, "Platt-calibrated", "s", GRAY[0])]:
        frac, mean_p = calibration_curve(yte, prob, n_bins=10, strategy="uniform")
        ax.plot(mean_p, frac, marker=mk, color=gc, lw=1.4, markersize=4, label=lab)
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction of positives")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(loc="upper left", fontsize=8, frameon=False)
    save_fig(fig, f"calibration_{sex_name}")

    # store for DCA
    dca_store[sex_name] = (yte_arr, prob_cal)

# ---- Tables: calibration + final summary ----
pd.DataFrame(calib_rows).to_csv(TAB / "calibration.csv", index=False)
pd.DataFrame(summary_rows).to_csv(TAB / "final_model_summary.csv", index=False)
print("\nFinal model summary:\n", pd.DataFrame(summary_rows).to_string(index=False))

# ---- Figure: Decision Curve Analysis (male/female, 2 panels) ----
def net_benefit(y, p, pt):
    yp = (p >= pt).astype(int)
    tp = ((yp == 1) & (y == 1)).sum(); fp = ((yp == 1) & (y == 0)).sum()
    n = len(y)
    return tp / n - (fp / n) * (pt / (1 - pt))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
pts = np.linspace(0.01, 0.60, 60)
for ax, sex_name in zip(axes, ["male", "female"]):
    y, p = dca_store[sex_name]
    prev = y.mean()
    nb_model = [net_benefit(y, p, t) for t in pts]
    nb_all = [prev - (1 - prev) * (t / (1 - t)) for t in pts]
    ax.plot(pts, nb_model, color=GRAY[0], lw=1.8, label="XGBoost")
    ax.plot(pts, nb_all, color=GRAY[2], lw=1.2, linestyle="--", label="Treat all")
    ax.axhline(0, color=GRAY[3], lw=1.0, linestyle=":", label="Treat none")
    ax.set_xlabel(f"Threshold probability — {sex_name.capitalize()}")
    ax.set_ylim(-0.05, max(nb_model) + 0.05)
    ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel("Net benefit")
save_fig(fig, "decision_curve_male_female")
print("Figures written: roc_{male,female}, calibration_{male,female}, decision_curve_male_female (png+pdf)")
print("Downstream artifacts saved to data/processed/.")
